# Figure 2: Cortical vs. Corticospinal Functional Gradients

This notebook reproduces **Figure 2** from the manuscript, with one code cell per subpanel (B–G). It assumes precomputed timecourses and connectivity matrices are available on disk, as configured via `config-results.json`.


## Config and imports

This cell loads the configuration file and imports all required libraries and utility functions.


In [ ]:
from utils import *

# Load analysis config
params = read_config('config-results.json')

# Schaefer SMC atlas
n_rois = 400
schaefer_dataset = datasets.fetch_atlas_schaefer_2018(n_rois=n_rois, resolution_mm=2)
schaefer_atlas = schaefer_dataset.maps
schaefer_labels = np.asarray(schaefer_dataset.labels, dtype=str)
smc_labels = []
for i,label in enumerate(schaefer_labels):
    if 'SomMot' in label:
        smc_labels.append(label)
smc_labels = pd.Series(smc_labels)

# Custom spinal cord atlas
sc_data = load_img(params["custom_sc_atlas"]).get_fdata()
sc_labels = open(params["custom_sc_labels"], 'r').read().splitlines()

## ROI selection and corticospinal FC

Load FC matrices, compute group-level FC, and build the SMC-spinal FC matrix used throughout Figure 2.


In [ ]:
# ------------------------------------------------------------------
# Load precomputed subject-level ROI-restricted FC matrices
# from params["save_fc_mats"] instead of recomputing FC upstream
# ------------------------------------------------------------------
fc_files = sorted(glob.glob(os.path.join(params["save_fc_mats"], "*.csv")))

if len(fc_files) == 0:
    raise FileNotFoundError(
        f"No FC csv files found in: {params['save_fc_mats']}"
    )

subFC_mats = []
sub_rois = None

for fpath in fc_files:
    df_fc = pd.read_csv(fpath, index_col=0)

    # Store ROI order from the first file and enforce consistency
    if sub_rois is None:
        sub_rois = df_fc.index.tolist()
    else:
        if df_fc.index.tolist() != sub_rois or df_fc.columns.tolist() != sub_rois:
            raise ValueError(
                f"ROI ordering mismatch in file: {fpath}"
            )

    subFC_mats.append(df_fc.values)

# Convert to array: n_subjects x n_rois x n_rois
subFC_mats = np.stack(subFC_mats, axis=0)

# ------------------------------------------------------------------
# Group-average sub-FC across subjects
# ------------------------------------------------------------------
mean_FC = np.mean(subFC_mats, axis=0)
subFC = mean_FC.copy()

# ------------------------------------------------------------------
# Rebuild ROI mapping for the already-saved FC matrices
# sub_rois should match the saved ROI-restricted FC order
# ------------------------------------------------------------------
rois_incl = sub_rois
rois_map = ['LH_SomMot' if 'LH' in roi else 'RH_SomMot' if 'RH' in roi and 'SomMot' in roi else roi.split(' ')[0]
            for roi in rois_incl]

# Identify cortical SMC indices within rois_incl
sm_idx = [j for j, roi in enumerate(rois_map) if 'SomMot' in roi]

# ------------------------------------------------------------------
# Extract cortical SMC-SMC block
# ------------------------------------------------------------------
cortical_FC = subFC[np.ix_(sm_idx, sm_idx)]

# ------------------------------------------------------------------
# Sparsify cortical FC by retaining strongest edges per row
# ------------------------------------------------------------------
sparsity_cortical = 0.9
cortical_sparse = np.array([
    row * (row > np.sort(row)[int(sparsity_cortical * len(row)) - 1])
    for row in cortical_FC
])

# ------------------------------------------------------------------
# Replace cortical block in the full subFC to obtain corticospinal FC
# ------------------------------------------------------------------
FC_cs = subFC.copy()
FC_cs[np.ix_(sm_idx, sm_idx)] = cortical_sparse

## Panel B – Corticospinal FC matrix

Plot the z-scored, sparsified Corticospinal FC matrix (62×110), ordered by hemisphere.


In [ ]:
# 1) Take 62x110 submatrix: rows = SMC, cols = all 110 (or also restrict cols if you want)
subFC_62x110 = subFC[sm_idx, :]   # shape (62, 110)

FC_z = subFC_62x110
# 2) Z-score and clip to [-1, 1]
FC_z = (subFC_62x110 - subFC_62x110.mean()) / subFC_62x110.std()
FC_z = np.clip(FC_z, -1, 1)

# 3) CSS-like diverging colormap
colors = ["#00a2ff", "#9ddff5", "#ffffff", "#ffbfdf", "#ff369b"]
cmap_css = LinearSegmentedColormap.from_list("css_fc_div", colors, N=256)

# 4) Figure size so each cell is ~square (62x110)
n_rows, n_cols = FC_z.shape
cell_size = 0.05
fig_size = (n_cols * cell_size, n_rows * cell_size)

fig, ax = plt.subplots(figsize=fig_size, dpi=300)

sns.heatmap(
    FC_z,
    cmap=cmap_css,
    square=False,          # not square now, matrix is rectangular
    cbar=True,
    linewidths=0.1,
    linecolor='white',
    xticklabels=False,
    yticklabels=False,
    ax=ax
) 

cbar = ax.collections[0].colorbar
cbar.set_label("Z-scored FC", rotation=270, labelpad=15)
# ---- add black separators ----
# 1) Horizontal LH/RH separator: between row 30 and 31
ax.hlines(30, xmin=0, xmax=62, colors='black', linewidth=0.5)
ax.vlines(30, ymin=0, ymax=62, colors='black', linewidth=0.4)

# 2) Vertical separators in spinal block:
# spinal columns start at index 62 (0-based: col 62..109)
spinal_start = 62
spinal_end = n_cols  # 110
step = 8

for c in range(spinal_start, spinal_end, step):
    ax.vlines(c, ymin=0, ymax=n_rows, colors='black', linewidth=0.2)


plt.tight_layout()
plt.show()
# out_path = os.path.join(
#     params["save_main_smc"],
#     "figure2_Corticospinal_FC.png"
# )
# fig.savefig(out_path, dpi=300, bbox_inches="tight")

## Panel C – Gradient explained variance

Compute diffusion map embedding on the sparsified corticospinal FC and plot the proportion of variance explained by the top five components.


In [ ]:
# Fit gradients for cortical-only and corticospinal FC
grads_cortical, lambdas_cortical, FC_cortical = fit_gradients(
    FC_cs,
    [rois_incl[j] for j in sm_idx],
    rois_incl,
    n_components=5,
    approach='dm',
    kernel='spearman',
    sparsity=0
)

grads_corticospinal, lambdas_corticospinal, FC_corticospinal = fit_gradients(
    FC_cs,
    [rois_incl[j] for j in sm_idx],
    rois_incl,
    rois_incl_y=rois_incl,
    n_components=5,
    approach='dm',
    kernel='spearman',
    sparsity=0
)

smc_grad = grads_cortical
smc_spinal_grad = grads_corticospinal
smc_grad_norm, smc_spinal_grad_aligned, disparity = align_procrustes_matlab(smc_spinal_grad, smc_grad, align_dims=2)

# project to array format with dimensions compatible for plotting in brainspace, and save in grad_arr/ folder
# this .npy file can then be loaded locally for plotting using brainspace's plot_hemispheres function
g_cortical_norm = project_brainspace_arr(smc_grad_norm[:,:2], [rois_incl[j] for j in sm_idx], fill=np.nan, nrois_schaefer=400)
# np.save(params["save_main_smc_npy"] + 'grad_SMC_2D_norm.npy', g_cortical_norm)

g_corticospinal_aligned = project_brainspace_arr(smc_spinal_grad_aligned[:,:2], [rois_incl[j] for j in sm_idx], fill=np.nan, nrois_schaefer=400)
# np.save(params["save_main_smc_npy"] + 'grad_SMC_spinal_2D_aligned.npy', g_corticospinal_aligned)

### Check Disparity
print(f"Disparity : grads_corticospinal, grads_cortical = {disparity:.3f}")

### Create grad dataframe
grads = {}
grads.update({
    f'SMC_Unaligned': smc_grad,
    f'SMC-spinal_Unaligned': smc_spinal_grad,
    f'SMC_Norm':smc_grad_norm, 
    f'SMC-spinal_Aligned':smc_spinal_grad_aligned
    })

for grad_type in ['SMC_Norm', 'SMC-spinal_Aligned']:
    if grads[grad_type].shape[0]!=77:
        g = np.empty((77,grads[grad_type].shape[1]))
        g[:] = np.nan
        i = 0
        for idx, label in enumerate(smc_labels):
            if label in (
                [f'7Networks_LH_SomMot_{id}' for id in range(1, 8)] +   # LH: 1–8
                [f'7Networks_RH_SomMot_{id}' for id in range(1, 9)]    # RH: 1–9
            ):
                continue
            g[idx, :] = grads[grad_type][i, :]
            i += 1
    else:
        g = grads[grad_type]

    

# Explained variance plot (cortical gradients)
x = np.arange(1, len(lambdas_corticospinal) + 1)

fig, ax = plt.subplots(figsize=(4, 3), dpi=300)
ax.plot(x, lambdas_corticospinal, marker='o', color='k', linewidth=1)
ax.set_xlabel('Component')
ax.set_ylabel('Explained variance (%)')
ax.axvspan(0.95, 2.05, color='grey', alpha=0.1)
ax.set_xticks(x)
ax.set_ylim(0, max(lambdas_corticospinal) * 1.1)
plt.tight_layout()
plt.show()
# out_path = os.path.join(
#     params["save_main_smc"],
#     "figure2_Corticospinal_ExpVar.png"
# )
# fig.savefig(out_path, dpi=300, bbox_inches="tight")


## Panel D – Corticospinal gradient maps

Project the first two Corticospinal gradients (G1, G2) onto the Schaefer cortical surface and save arrays for external visualization (e.g. BrainSpace).


In [ ]:
# Panel C: project G1 and G2 onto Schaefer surface

smc_grad = grads_cortical
smc_spinal_grad = grads_corticospinal

# Align spinal gradients to cortical via Procrustes
smc_grad_norm, smc_spinal_grad_aligned, disparity = align_procrustes_matlab(
    smc_spinal_grad,
    smc_grad,
    align_dims=2
)

# Project to full Schaefer400 for surface plotting (BrainSpace)
g_cortical_norm = project_brainspace_arr(
    smc_grad_norm[:, :2],
    [rois_incl[j] for j in sm_idx],
    fill=np.nan,
    nrois_schaefer=400
)
# np.save(params["save_main_smc_npy"] + 'grad_SMC_2D_norm.npy', g_cortical_norm)

g_corticospinal_aligned = project_brainspace_arr(
    smc_spinal_grad_aligned[:, :2],
    [rois_incl[j] for j in sm_idx],
    fill=np.nan,
    nrois_schaefer=400
)
# np.save(params["save_main_smc_npy"] + 'grad_SMC_spinal_2D_aligned.npy', g_corticospinal_aligned)

print(f"Disparity (corticospinal vs cortical gradients): {disparity:.3f}")


## Panel E – G1 vs G2 scatter (no clusters)

Scatter plot of normalized Corticospinal gradient values (G1 vs G2) across all parcels, without cluster assignment.


In [ ]:
# Build SMC gradient dataframe
smc_spinal_grad_df = pd.DataFrame({
    'G1': smc_spinal_grad[:, 0],
    'G2': smc_spinal_grad[:, 1],
    'RoI': [rois_incl[j] for j in sm_idx]
})

# z-score across all SMC parcels
smc_spinal_grad_df['z_G1'] = (smc_spinal_grad_df['G1'] - smc_spinal_grad_df['G1'].mean()) / smc_spinal_grad_df['G1'].std()
smc_spinal_grad_df['z_G2'] = (smc_spinal_grad_df['G2'] - smc_spinal_grad_df['G2'].mean()) / smc_spinal_grad_df['G2'].std()

fig, ax = plt.subplots(figsize=(4, 4), dpi=300)
sns.scatterplot(
    data=smc_spinal_grad_df,
    x='z_G1', y='z_G2',
    color='black',
    alpha=0.6,
    s=50,
    edgecolor='k',
    ax=ax,
    legend=False
)
ax.axhline(0, color='grey', linestyle='--', linewidth=1)
ax.axvline(0, color='grey', linestyle='--', linewidth=1)
ax.set_xlabel('z(G1)')
ax.set_ylabel('z(G2)')
ax.set_title('Corticospinal (smc-spinal) gradients: z-scored G1 vs G2 (no clusters)')
plt.tight_layout()
plt.show()
# out_path = os.path.join(
#     params["save_main_smc"],
#     "figure2_G1G2_scatter_no_clusters.png"
# )
# fig.savefig(out_path, dpi=300, bbox_inches="tight")


## Panel F – G1 vs G2 scatter with functional clustering (K = 4)

Scatter plot of normalized Cortical (smc) Corticospinal (smc-spinal) gradient values (G1 vs G2) across all parcels, with cluster assignment. Th ecluster assignment is computed on cortical gradient space and this assignment is enforced on corticospinal gradient space.

In [ ]:
# Dataframes and clustering

lh_labels = pd.read_csv(params["dist_file_L"])
rh_labels = pd.read_csv(params["dist_file_R"])

som_labels_L = lh_labels["roi_label"].tolist()
som_labels_R = rh_labels["roi_label"].tolist()
som_labels = som_labels_L + som_labels_R

lh_dist = lh_labels["cum_distance_mm"].tolist()
rh_dist = rh_labels["cum_distance_mm"].tolist()
dist = lh_dist + rh_dist

GRAD_TYPES = ["SMC_Norm", "SMC-spinal_Aligned"]
K = 4
RANDOM_STATE = 10

def cluster_and_analyze(df_all, grad_type, k=K, random_state=RANDOM_STATE, cluster_feature="G2"):
    x_var = "Dist"
    y_var = cluster_feature
    df_sub = df_all[[x_var, y_var, "RoI", "G1", "G2"]].dropna().copy()
    if df_sub.empty:
        return None
    X = df_sub[[x_var, y_var]].values
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    spectral = SpectralClustering(
        n_clusters=k,
        affinity="rbf",
        gamma=1.0,
        random_state=random_state,
    )
    labels = spectral.fit_predict(X_scaled)
    df_sub["cluster"] = labels
    label_map = dict(zip(df_sub["RoI"], df_sub["cluster"]))
    df_all = df_all.copy()
    df_all["cluster_smc"] = df_all["RoI"].map(label_map)
    df_all = df_all[~df_all["cluster_smc"].isna()].copy()
    df_all["cluster_smc"] = df_all["cluster_smc"].astype(int)
    df_all["z_Dist"] = (df_all["Dist"] - df_all["Dist"].mean()) / df_all["Dist"].std()
    df_all["z_G1"] = (df_all["G1"] - df_all["G1"].mean()) / df_all["G1"].std()
    df_all["z_G2"] = (df_all["G2"] - df_all["G2"].mean()) / df_all["G2"].std()
    return df_all

df_all_dict = {}
for grad_type in GRAD_TYPES:
    g1 = grads[grad_type][:, 0]
    g2 = grads[grad_type][:, 1]
    df_all = pd.DataFrame(
        {"G1": g1, "G2": g2, "RoI": som_labels, "Dist": dist}
    )
    df_all_dict[grad_type] = cluster_and_analyze(
        df_all,
        grad_type,
        k=K,
        random_state=RANDOM_STATE,
        cluster_feature="G2",
    )

df_smc = df_all_dict["SMC_Norm"][["RoI", "G1", "G2", "z_Dist"]].dropna().copy()
df_spin = df_all_dict["SMC-spinal_Aligned"][["RoI", "G1", "G2", "z_Dist"]].dropna().copy()

df_smc = df_smc.sort_values("RoI").reset_index(drop=True)
df_spin = df_spin.sort_values("RoI").reset_index(drop=True)

common_rois = sorted(set(df_smc["RoI"]) & set(df_spin["RoI"]))
df_smc = df_smc[df_smc["RoI"].isin(common_rois)].sort_values("RoI").reset_index(drop=True)
df_spin = df_spin[df_spin["RoI"].isin(common_rois)].sort_values("RoI").reset_index(drop=True)

df_smc["z_G1"] = (df_smc["G1"] - df_smc["G1"].mean()) / df_smc["G1"].std()
df_smc["z_G2"] = (df_smc["G2"] - df_smc["G2"].mean()) / df_smc["G2"].std()

X_smc = df_smc[["z_G1", "z_G2"]].values
spectral = SpectralClustering(
    n_clusters=K,
    affinity="rbf",
    gamma=1.0,
    random_state=RANDOM_STATE,
)
df_smc["cluster_smc"] = spectral.fit_predict(X_smc)
df_spin["cluster_smc"] = df_smc["cluster_smc"].values

df_all_smc = df_all_dict["SMC_Norm"].copy()
df_all_spinal = df_all_dict["SMC-spinal_Aligned"].copy()

cluster_map_smc = dict(zip(df_smc["RoI"], df_smc["cluster_smc"]))
cluster_map_spin = dict(zip(df_spin["RoI"], df_spin["cluster_smc"]))

df_all_smc["cluster_smc"] = df_all_smc["RoI"].map(cluster_map_smc)
df_all_spinal["cluster_smc"] = df_all_spinal["RoI"].map(cluster_map_spin)

df_all_smc = df_all_smc.dropna(subset=["cluster_smc"]).copy()
df_all_spinal = df_all_spinal.dropna(subset=["cluster_smc"]).copy()

df_all_smc["cluster_smc"] = df_all_smc["cluster_smc"].astype(int)
df_all_spinal["cluster_smc"] = df_all_spinal["cluster_smc"].astype(int)

for df in [df_all_smc, df_all_spinal]:
    df["z_G1"] = (df["G1"] - df["G1"].mean()) / df["G1"].std()
    df["z_G2"] = (df["G2"] - df["G2"].mean()) / df["G2"].std()

cluster_hex = {0: "#219ebc", 1: "#023047", 2: "#ffb703", 3: "#fb8500"}
palette = [cluster_hex[c] for c in sorted(cluster_hex.keys())]

# Visualization 1: SMC_Norm, cluster colors
fig, ax = plt.subplots(figsize=(6, 6), dpi=300)
sns.scatterplot(
    data=df_all_smc,
    x="z_G1",
    y="z_G2",
    hue="cluster_smc",
    palette=palette,
    alpha=0.7,
    s=200,
    edgecolor="k",
    ax=ax,
    legend=False,
)
ax.axhline(0, color="grey", linestyle="--", linewidth=1)
ax.axvline(0, color="grey", linestyle="--", linewidth=1)
ax.set_title("SMC_Norm: z-scored G1–G2 (clusters)")
ax.set_xlabel("z(G1)")
ax.set_ylabel("z(G2)")
plt.tight_layout()
plt.show()
out_path = os.path.join(
    params["save_main_smc"],
    "figure2_SMC_GradientScatter_K4.png"
)
fig.savefig(out_path, dpi=300, bbox_inches="tight")

# Visualization 2: SMC-spinal_Aligned, cluster colors
fig, ax = plt.subplots(figsize=(6, 6), dpi=300)
sns.scatterplot(
    data=df_all_spinal,
    x="z_G1",
    y="z_G2",
    hue="cluster_smc",
    palette=palette,
    alpha=0.7,
    s=200,
    edgecolor="k",
    ax=ax,
)
ax.axhline(0, color="grey", linestyle="--", linewidth=1)
ax.axvline(0, color="grey", linestyle="--", linewidth=1)
ax.set_title("SMC-spinal_Aligned: z-scored G1–G2 (clusters)")
ax.set_xlabel("z(G1)")
ax.set_ylabel("z(G2)")
plt.tight_layout()
plt.show()
out_path = os.path.join(
    params["save_main_smc"],
    "figure2_SMC_Spinal_GradientScatter_K4.png"
)
fig.savefig(out_path, dpi=300, bbox_inches="tight")

## List of cluster assigment following spectral clustering in cortical gradient space for K = 4; This assigment is stored
## and later used in enrichment assigment for each Himunculus derived functional class (FA, TRUNK, LL, UL). 
cluster_col = "cluster_smc"
df_for_print_smc = df_all_smc[["RoI", cluster_col]].dropna().copy()
df_for_print_spinal = df_all_spinal[["RoI", cluster_col]].dropna().copy()

clusters = sorted(df_for_print_smc[cluster_col].unique())

rows = []
for c in clusters:
    rois_smc = df_for_print_smc.loc[df_for_print_smc[cluster_col] == c, "RoI"]
    for roi in rois_smc:
        rows.append(["SMC_Norm", c, roi])
    rois_spinal = df_for_print_spinal.loc[df_for_print_spinal[cluster_col] == c, "RoI"]
    for roi in rois_spinal:
        rows.append(["SMC-spinal_Aligned", c, roi])

cluster_table = pd.DataFrame(rows, columns=["Map", "Cluster", "RoI"])
#cluster_table.to_csv("cluster_composition.csv", index=False)

print(cluster_table.head(30))

## Panel G – Reorganization

Per-cluster (LL, UL, TRUNK, FA) contour plots of Cortical (smc-smc) and Corticospinal (smc-spinal) scatter, with boxplots of cluster radii across parcels for each functional class. The cluster radii metric serves as a measures of reorganization (dispersion/separation/expansion) of the functional cluster with two groups. 

In [ ]:
from plotnine import (
    ggplot, aes, geom_point, geom_boxplot, geom_violin, geom_line,
    position_jitter, scale_color_manual, scale_fill_manual,
    xlab, ylab, coord_cartesian, theme_classic, theme
)

# =========================================================
# 0) Preprocessing and configuration
# =========================================================
# Assumes df_all_smc and df_all_spinal exist with:
# ['RoI','G1','G2','cluster_smc'] and matched RoIs/clusters.

df_smc = df_all_smc.copy()
df_spin = df_all_spinal.copy()

# Hemisphere labels
for df in [df_smc, df_spin]:
    df['Hemisphere'] = np.where(df['RoI'].str.contains('RH'), 'RH', 'LH')

# z-scores within each map
for df in [df_smc, df_spin]:
    df['z_G1'] = (df['G1'] - df['G1'].mean()) / df['G1'].std()
    df['z_G2'] = (df['G2'] - df['G2'].mean()) / df['G2'].std()

# Integer clusters and color palette
df_smc['cluster_smc'] = df_smc['cluster_smc'].astype(int)
df_spin['cluster_smc'] = df_spin['cluster_smc'].astype(int)

clusters = sorted(df_smc['cluster_smc'].unique())
marker_map_hemi = {'LH': 'o', 'RH': '^'}
marker_all = 'o'

cluster_hex = {
    0: '#219ebc',
    1: '#023047',
    2: '#ffb703',
    3: '#fb8500',
}
palette = [cluster_hex[c] for c in sorted(cluster_hex.keys())]

# =========================================================
# 1) Helper plotting functions – centroids and KDE envelopes
# =========================================================

def plot_centroids_panel(ax, df_smc, df_spin, use_hemi_markers=False,
                         title='SMC vs Spinal: z(G1)–z(G2) centroids'):
    """Shared z(G1)–z(G2) space: SMC vs spinal points + centroids."""

    # SMC points (filled)
    for c in clusters:
        df_c = df_smc[df_smc['cluster_smc'] == c]
        ax.scatter(
            df_c['z_G1'],
            df_c['z_G2'],
            s=80,
            color=palette[c],
            alpha=0.25,
            edgecolor='none',
        )

    # Spinal points (outlined, optional LH/RH markers)
    if use_hemi_markers:
        for hemi in ['LH', 'RH']:
            for c in clusters:
                df_c = df_spin[(df_spin['cluster_smc'] == c) &
                               (df_spin['Hemisphere'] == hemi)]
                ax.scatter(
                    df_c['z_G1'],
                    df_c['z_G2'],
                    s=80,
                    facecolors='none',
                    edgecolors=palette[c],
                    marker=marker_map_hemi[hemi],
                    linewidth=1.5,
                )
    else:
        for c in clusters:
            df_c = df_spin[df_spin['cluster_smc'] == c]
            ax.scatter(
                df_c['z_G1'],
                df_c['z_G2'],
                s=80,
                facecolors='none',
                edgecolors=palette[c],
                marker=marker_all,
                linewidth=1.5,
            )

    # Centroids per cluster
    for c in clusters:
        df_c_smc = df_smc[df_smc['cluster_smc'] == c]
        df_c_spin = df_spin[df_spin['cluster_smc'] == c]

        mu_g1_smc = df_c_smc['z_G1'].mean()
        mu_g2_smc = df_c_smc['z_G2'].mean()
        mu_g1_spin = df_c_spin['z_G1'].mean()
        mu_g2_spin = df_c_spin['z_G2'].mean()

        ax.scatter(
            mu_g1_smc,
            mu_g2_smc,
            s=220,
            color=palette[c],
            alpha=0.5,
            edgecolor='k',
            linewidth=1.0,
            zorder=3,
        )
        ax.scatter(
            mu_g1_spin,
            mu_g2_spin,
            s=220,
            color=palette[c],
            alpha=1.0,
            facecolors='none',
            edgecolor='k',
            linewidth=1.5,
            marker='o',
            zorder=4,
        )

    ax.axhline(0, color='grey', linestyle='--', linewidth=1)
    ax.axvline(0, color='grey', linestyle='--', linewidth=1)
    ax.set_xlabel('z(G1)')
    ax.set_ylabel('z(G2)')
    ax.set_title(title)

    handles = [
        plt.Line2D([0], [0], marker='o', color='w',
                   markerfacecolor='grey', alpha=0.25, markeredgecolor='none',
                   markersize=7, label='SMC points'),
        plt.Line2D([0], [0], marker='o', color='w',
                   markerfacecolor='none', markeredgecolor='grey',
                   markersize=7, label='Spinal points'),
        plt.Line2D([0], [0], marker='o', color='w',
                   markerfacecolor='grey', alpha=0.4, markeredgecolor='k',
                   markersize=9, label='SMC centroids'),
        plt.Line2D([0], [0], marker='o', color='w',
                   markerfacecolor='none', markeredgecolor='k',
                   markersize=9, label='Spinal centroids'),
    ]
    ax.legend(handles=handles, loc='best')


def plot_per_cluster_kde_panels(df_smc, df_spin, use_hemi_markers=False):
    """Per-cluster KDE + points (one figure per cluster)."""

    levels = 8

    for c in clusters:
        color = palette[c]
        df_smc_c = df_smc[df_smc['cluster_smc'] == c]
        df_spin_c = df_spin[df_spin['cluster_smc'] == c]

        fig, ax = plt.subplots(figsize=(5, 5), dpi=300)

        if len(df_smc_c) >= 3:
            sns.kdeplot(
                ax=ax,
                data=df_smc_c,
                x='z_G1',
                y='z_G2',
                levels=levels,
                fill=True,
                linestyles='-',
                color=None,
                cmap=sns.dark_palette(color, as_cmap=True, reverse=True),
                alpha=0.6,
                linewidths=1.0,
            )

        if len(df_spin_c) >= 3:
            sns.kdeplot(
                ax=ax,
                data=df_spin_c,
                x='z_G1',
                y='z_G2',
                levels=levels,
                fill=True,
                linestyles='--',
                color=None,
                cmap=sns.light_palette(color, as_cmap=True, reverse=True),
                alpha=0.3,
                linewidths=1.0,
            )

        ax.scatter(
            df_smc_c['z_G1'],
            df_smc_c['z_G2'],
            s=200,
            facecolors=color,
            alpha=0.6,
            edgecolor='k',
            marker='o',
            linewidth=1.5,
            zorder=3,
        )

        if use_hemi_markers:
            for hemi in ['LH', 'RH']:
                df_h = df_spin_c[df_spin_c['Hemisphere'] == hemi]
                ax.scatter(
                    df_h['z_G1'],
                    df_h['z_G2'],
                    s=200,
                    facecolors='none',
                    edgecolors=color,
                    marker=marker_map_hemi[hemi],
                    linewidth=1.5,
                    zorder=3,
                )
        else:
            ax.scatter(
                df_spin_c['z_G1'],
                df_spin_c['z_G2'],
                s=200,
                facecolors='none',
                edgecolors=color,
                marker=marker_all,
                linewidth=1.5,
                zorder=3,
            )

        ax.axhline(0, color='grey', linestyle='--', linewidth=1)
        ax.axvline(0, color='grey', linestyle='--', linewidth=1)
        ax.set_title(f'Cluster {c}: SMC vs spinal')
        ax.set_xlabel('z(G1)')
        ax.set_ylabel('z(G2)')

        handles = [
            plt.Line2D([0], [0], color=color, linewidth=2, linestyle='-',
                       label='SMC KDE'),
            plt.Line2D([0], [0], color=color, linewidth=2, linestyle='--',
                       label='Spinal KDE'),
            plt.Line2D([0], [0], marker='o', color='w',
                       markerfacecolor=color, alpha=0.5, markeredgecolor='k',
                       markersize=8, label='SMC points'),
            plt.Line2D([0], [0], marker='o', color='w',
                       markerfacecolor='none', markeredgecolor=color,
                       markersize=8, label='Spinal points'),
        ]
        fig.legend(handles=handles, loc='center left', bbox_to_anchor=(1.02, 0.5))
        plt.tight_layout()
        plt.show()

# =========================================================
# 2) ROI dispersion: radii and boxplots
# =========================================================

# Align SMC and spinal by RoI
df_smc_aligned = df_smc.sort_values('RoI').reset_index(drop=True)
df_spin_aligned = df_spin.sort_values('RoI').reset_index(drop=True)

df_pair = pd.merge(
    df_smc_aligned[['RoI', 'z_G1', 'z_G2', 'cluster_smc']].rename(
        columns={'z_G1': 'z_G1_smc', 'z_G2': 'z_G2_smc'}
    ),
    df_spin_aligned[['RoI', 'z_G1', 'z_G2']].rename(
        columns={'z_G1': 'z_G1_spin', 'z_G2': 'z_G2_spin'}
    ),
    on='RoI',
    how='inner',
)
df_pair = df_pair.copy()
clusters = sorted(df_pair['cluster_smc'].astype(int).unique())
df_pair['cluster_smc'] = df_pair['cluster_smc'].astype(int)

# Per-ROI radii (distance to within-cluster centroid)
radius_rows = []
for c in clusters:
    df_c = df_pair[df_pair['cluster_smc'] == c].copy()

    mu_smc = np.array([df_c['z_G1_smc'].mean(), df_c['z_G2_smc'].mean()])
    mu_spin = np.array([df_c['z_G1_spin'].mean(), df_c['z_G2_spin'].mean()])

    r_smc = np.linalg.norm(df_c[['z_G1_smc', 'z_G2_smc']].values - mu_smc, axis=1)
    r_spin = np.linalg.norm(df_c[['z_G1_spin', 'z_G2_spin']].values - mu_spin, axis=1)

    for roi, rs, rp in zip(df_c['RoI'], r_smc, r_spin):
        radius_rows.append({
            'RoI': roi,
            'cluster': c,
            'radius_smc': rs,
            'radius_spin': rp,
            'delta_radius': rp - rs,
        })

radius_df = pd.DataFrame(radius_rows)
radius_df['cluster'] = radius_df['cluster'].astype(int)

# Cluster-wise paired t-tests
summary_rows = []
for c in clusters:
    df_c = radius_df[radius_df['cluster'] == c]
    rs = df_c['radius_smc'].values
    rp = df_c['radius_spin'].values

    if len(df_c) >= 3:
        t_stat, p_val = stats.ttest_rel(rp, rs, nan_policy='omit')
    else:
        t_stat, p_val = np.nan, np.nan

    summary_rows.append({
        'cluster': c,
        'n_ROI': len(df_c),
        'mean_radius_SMC': rs.mean(),
        'mean_radius_Spinal': rp.mean(),
        'mean_delta_radius': (rp - rs).mean(),
        't_stat': t_stat,
        'p_val': p_val,
    })

summary_df = pd.DataFrame(summary_rows)
print("Cluster-wise radius summary:")
print(summary_df)

# Global paired t-test across RoIs
wide_roi = radius_df.pivot_table(index='RoI', values=['radius_smc', 'radius_spin']).dropna()
t_stat_global, p_val_global = stats.ttest_rel(
    wide_roi['radius_spin'], wide_roi['radius_smc'], nan_policy='omit'
)
print(f"\nGlobal paired t-test: t = {t_stat_global:.3f}, p = {p_val_global:.4f}")

# Long format for radius boxplots
rows = []
for _, row in radius_df.iterrows():
    rows.append({'RoI': row['RoI'], 'cluster': row['cluster'],
                 'map': 'SMC',    'radius': row['radius_smc']})
    rows.append({'RoI': row['RoI'], 'cluster': row['cluster'],
                 'map': 'Spinal', 'radius': row['radius_spin']})
long_df = pd.DataFrame(rows)
long_df['cluster'] = long_df['cluster'].astype(int)
long_df['map'] = long_df['map'].astype(str)

def plot_radius_boxplots(long_df, clusters):
    """Per-cluster SMC vs spinal radius distributions (violin + boxplot)."""
    my_hex = ["#2a9d8f", "#e76f51"]

    for c in clusters:
        df_c = long_df[long_df['cluster'] == c].copy()
        df_c['x_center'] = 1.0

        offset_map = {'SMC': -1, 'Spinal': 1}
        df_c['x_point'] = df_c['x_center'] + df_c['map'].map(lambda m: 0.08 * offset_map[m])
        df_c['x_line'] = df_c['x_center'] + df_c['map'].map(lambda m: 0.08 * offset_map[m])
        df_c['x_box'] = df_c['x_center'] + 0.20 * df_c['map'].map(offset_map)
        df_c['x_violin'] = df_c['x_center'] + 0.30 * df_c['map'].map(offset_map)
        plots = {}
        p = (
            ggplot(df_c)
            + geom_violin(
                aes(x='x_violin', y='radius', fill='map'),
                style="left-right",
                trim=True,
                alpha=0.6,
                color=None,
                size=1.0,
            )
            + geom_boxplot(
                aes(x='x_box', y='radius', fill='map'),
                width=0.1,
                outlier_alpha=0,
                alpha=0.8,
                size=1.0,
            )
            + geom_point(
                aes(x='x_point', y='radius', fill='map'),
                size=3.0,
                stroke=0.8,
                show_legend=False,
            )
            + geom_line(
                aes(x='x_line', y='radius', group='RoI'),
                color="gray",
                size=1.0,
                alpha=0.5,
            )
            + scale_fill_manual(values={'SMC': my_hex[0], 'Spinal': my_hex[1]})
            + xlab(f'Cluster {c}')
            + theme_classic()
            + theme(
                figure_size=(4, 4),
                axis_text_x=None,
                axis_ticks_major_x=None,
            )
        )
        plots[int(c)] = p
    return plots


# =========================================================
# 3) Figure calls (four visualization types)
# =========================================================

# 1) Centroids + points (global)
fig, ax = plt.subplots(figsize=(6, 6), dpi=300)
plot_centroids_panel(ax, df_smc=df_smc, df_spin=df_spin, use_hemi_markers=False)
plt.tight_layout()
plt.show()

# 3) Per-cluster KDE + points
plot_per_cluster_kde_panels(df_smc=df_smc, df_spin=df_spin, use_hemi_markers=False)

# 4) ROI dispersion boxplots (per cluster)
plots = plot_radius_boxplots(long_df=long_df, clusters=clusters)

In [ ]:
plots[3]